# Single Pendulum Neural Network

In [31]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pprint

All inputs for the AI
Cart Position - where the cart is
Cart Velocity - to stop overshooting and knowing what happens to cart between each frame instead of it learning it itself
Pendulum X - Gonna have x and y instead of $\theta$ since 0 doesn't smoothly go to 359 even in radian while x and y even if they have doubles
Pendulum Y - Never gonna guess but same as X
Pendulum Angular Velocity - Momentum of the pendulum

## Neat Algorithm

In [90]:
g = 9.81
l = 1.0
m = 1.0
dt = 0.02
n = 100 # Gonna be how many pendulums I'm going to spawn

### Representing as a DAG

In [91]:
genome = {
    "nodes": {
        0: {"type": "INPUT", "act": "identity"},  # x
        1: {"type": "INPUT", "act": "identity"},  # v
        2: {"type": "INPUT", "act": "identity"},  # θ
        3: {"type": "INPUT", "act": "identity"},  # ω
        4: {"type": "OUTPUT", "act": "tanh"},  # Motor Force (-1 to 1)
    },
    "connections": {
        # Link from angle to force
        101: {"in": 2, "out": 4, "weight": 0.5, "enabled": True},
        # Link from angular velocity to force
        102: {"in": 3, "out": 4, "weight": -0.5, "enabled": True},
    },
}

In [108]:
genome = {
    "nodes": {
        0: {"type": "INPUT", "act": "identity"},  # x
        1: {"type": "INPUT", "act": "identity"},  # v
        2: {"type": "INPUT", "act": "identity"},  # θ
        3: {"type": "INPUT", "act": "identity"},  # ω
        4: {"type": "OUTPUT", "act": "tanh"},  # Motor Force
        5: {"type": "HIDDEN", "act": "identity"},  # Hidden Node
        6: {"type": "HIDDEN", "act": "identity"},  # Hidden Node
    },
    "connections": {
        101: {"in": 2, "out": 4, "weight": 0.5, "enabled": False},  # Disabled
        103: {"in": 2, "out": 5, "weight": 1.0, "enabled": True},
        104: {"in": 5, "out": 4, "weight": 0.4, "enabled": True},
        102: {"in": 3, "out": 4, "weight": -0.5, "enabled": True},
        105: {"in": 1, "out": 5, "weight": -0.2, "enabled": True},
        106: {"in": 1, "out": 6, "weight": -0.2, "enabled": True},
        107: {"in": 6, "out": 4, "weight": -0.2, "enabled": True},
    },
}

In [109]:
def graph_dag(genome):
    G = nx.DiGraph()

    for node_id, data in genome["nodes"].items():
        G.add_node(node_id, type=data["type"], act=data["act"]) # With info to be displayed

    for conn in genome["connections"].values():
        if conn["enabled"]:
            G.add_edge(conn["in"], conn["out"], weight=conn["weight"])

    for node_id, data in genome["nodes"].items():
        if data["type"] == "INPUT":
            G.nodes[node_id]["layer"] = 0
        elif data["type"] == "OUTPUT":
            G.nodes[node_id]["layer"] = 2
        else:
            G.nodes[node_id]["layer"] = 1

    pos = nx.multipartite_layout(G, subset_key="layer") # This is going to be the positions of these

    edge_traces = []
    for edge in G.edges(data=True):
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        weight = edge[2]["weight"]

        color = "rgba(50, 100, 250, 0.8)" if weight > 0 else "rgba(250, 50, 50, 0.8)"

        trace = go.Scatter(
            x=[x0, x1, None],
            y=[y0, y1, None],
            line=dict(width=abs(weight) * 5, color=color),
            hoverinfo="none",
            mode="lines",
        )
        edge_traces.append(trace)

    node_x, node_y, node_hover = [], [], []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_hover.append(
            f"Node {node}<br>Type: {G.nodes[node]['type']}<br>Act: {G.nodes[node]['act']}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=[f"N{n}" for n in G.nodes()],
        textposition="top center",
        hoverinfo="text",
        hovertext=node_hover,
        marker=dict(
            size=25,
            color=[
                "#636EFA" if G.nodes[n]["type"] == "INPUT" else "#EF553B"
                for n in G.nodes()
            ],
            line_width=2,
        ),
    )

    fig = go.Figure(
        data=edge_traces + [node_trace],
        layout=go.Layout(
            title="Evolved Pendulum Controller (NEAT)",
            showlegend=False,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white",
        ),
    )
    fig.show()

In [110]:
graph_dag(genome)

### Solve DAG

In [111]:
inputs = np.array([0.5, 0.1, 0.2, 0.3])  # Initial values for [x, v, θ, ω]

#### Kahns

So far I know we can do Kahn's Algorithm or use DFS. Gonna do Kahn's for now.

In [112]:
# Get Topological Order
nodes = genome["nodes"]
connections = genome["connections"]

in_degree = {node_id: 0 for node_id in nodes}
for conn in connections.values():
    # Basically for every connection we add to the in_degree of every node or how many connection go into node
    if conn["enabled"]:
        in_degree[conn["out"]] += 1

# So we do a queue where if the degree is 0 then no connection go into node like the INPUT nodes
queue = [node_id for node_id, degree in in_degree.items() if degree == 0]
topo_order = []

while queue:
    current = queue.pop(0)
    topo_order.append(current)

    # For every connection going out of the current node, we reduce the in_degree
    for conn in connections.values():
        # So in basically means starting so if it starts from the current node then we basically say it's use up and the connection is broken
        # Since once we have the value and let's say next item has 0 connection then this is the only connection but if it has 1 then we wait till that one is also broken
        if conn["enabled"] and conn["in"] == current:
            in_degree[conn["out"]] -= 1
            # If it's 0 then we add it to the queue
            if in_degree[conn["out"]] == 0:
                queue.append(conn["out"])

print("Topological Order:", topo_order)

Topological Order: [0, 1, 2, 3, 6, 5, 4]


In [113]:
genome["connections"]

{101: {'in': 2, 'out': 4, 'weight': 0.5, 'enabled': False},
 103: {'in': 2, 'out': 5, 'weight': 1.0, 'enabled': True},
 104: {'in': 5, 'out': 4, 'weight': 0.4, 'enabled': True},
 102: {'in': 3, 'out': 4, 'weight': -0.5, 'enabled': True},
 105: {'in': 1, 'out': 5, 'weight': -0.2, 'enabled': True},
 106: {'in': 1, 'out': 6, 'weight': -0.2, 'enabled': True},
 107: {'in': 6, 'out': 4, 'weight': -0.2, 'enabled': True}}

In [114]:
node_values = {node_id: 0.0 for node_id in genome["nodes"]}

for i, val in enumerate(inputs):
    node_values[i] = val

for node_id in topo_order:
    node_data = genome["nodes"][node_id]

    # if node_data["type"] != "INPUT":
    #     raw_val = node_values[node_id]
    #     if node_data["act"] == "tanh":
    #         node_values[node_id] = np.tanh(raw_val) # When we do tanh all connection into are already done so tanh is done at the end not inbetween
    #     elif node_data["act"] == "identity":
    #         node_values[node_id] = raw_val

    # Propogate forward
    for conn in genome["connections"].values():
        if conn["enabled"] and conn["in"] == node_id: # So starts from this node
            node_values[conn["out"]] += node_values[node_id] * conn["weight"] # We add to the output node

print("Motor Activation:", node_values[4])

Motor Activation: -0.074


In [115]:
# Hide tanh for now t
node_values

{0: np.float64(0.5),
 1: np.float64(0.1),
 2: np.float64(0.2),
 3: np.float64(0.3),
 4: np.float64(-0.074),
 5: np.float64(0.18),
 6: np.float64(-0.020000000000000004)}

#### Matrix

In [116]:
dag_matrix = np.zeros((len(genome["nodes"]), len(genome["nodes"])))
dag_sol = np.zeros(len(genome["nodes"]))

for i in range(len(genome["nodes"])):
    if genome["nodes"][i]["type"] == "INPUT":
        dag_matrix[i, i] = 1.0  # Identity for input nodes
        dag_sol[i] = inputs[i]
    else:
        dag_matrix[i, i] = -1.0
        dag_sol[i] = 0.0

print("Initial DAG Matrix:\n", dag_matrix)
print("Initial DAG Solution:\n", dag_sol)

for conn in genome["connections"].values():
    if conn["enabled"]:
        dag_matrix[conn["out"], conn["in"]] = conn["weight"]

print("DAG Adjacency Matrix:\n", dag_matrix)
print("DAG Solution Matrix:\n", dag_sol)

node_values = np.linalg.solve(dag_matrix, dag_sol)
print(node_values)

Initial DAG Matrix:
 [[ 1.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  0. -1.  0.  0.]
 [ 0.  0.  0.  0.  0. -1.  0.]
 [ 0.  0.  0.  0.  0.  0. -1.]]
Initial DAG Solution:
 [0.5 0.1 0.2 0.3 0.  0.  0. ]
DAG Adjacency Matrix:
 [[ 1.   0.   0.   0.   0.   0.   0. ]
 [ 0.   1.   0.   0.   0.   0.   0. ]
 [ 0.   0.   1.   0.   0.   0.   0. ]
 [ 0.   0.   0.   1.   0.   0.   0. ]
 [ 0.   0.   0.  -0.5 -1.   0.4 -0.2]
 [ 0.  -0.2  1.   0.   0.  -1.   0. ]
 [ 0.  -0.2  0.   0.   0.   0.  -1. ]]
DAG Solution Matrix:
 [0.5 0.1 0.2 0.3 0.  0.  0. ]
[ 0.5    0.1    0.2    0.3   -0.074  0.18  -0.02 ]


It works without tanh we could apply at the end but if we want hidden nodes to have tanh then it might not work. Don't think it's worth it since unlike a neural network where every layers creates a full matrix this creates a sparse matrix. And if we have nodes that skip layers like we have for node 4 here where 3 in layers 1 and 5 from layer 2 connects to it. We need a bigger matrix with all past layers for every next layers we go to. But if we need to parallize we could do a tensor or a bigger 2D array but we now need to fit the matrix into the biggest matrix and have unncescary memory and even more sparse. So gonna go with the loop for now then try to speed up. Also could try to do matrix mult at the end since it might be faster and I could update more.

#### Matrix With Non Linear

In [ ]:
node_values = {node_id: 0.0 for node_id in genome["nodes"]}

for i, val in enumerate(inputs):
    node_values[i] = val

for node_id in topo_order:
    node_data = genome["nodes"][node_id]

    if node_data["type"] != "INPUT":
        raw_val = node_values[node_id]
        if node_data["act"] == "tanh":
            node_values[node_id] = np.tanh(raw_val) # When we do tanh all connection into are already done so tanh is done at the end not inbetween
        elif node_data["act"] == "identity":
            node_values[node_id] = raw_val

    # Propogate forward
    for conn in genome["connections"].values():
        if conn["enabled"] and conn["in"] == node_id:  # So starts from this node
            node_values[conn["out"]] += (
                node_values[node_id] * conn["weight"]
            )  # We add to the output node
print(node_values)
print("Motor Activation:", node_values[4])

{0: np.float64(0.5), 1: np.float64(0.1), 2: np.float64(0.2), 3: np.float64(0.3), 4: np.float64(-0.0738652205465518), 5: np.float64(0.18), 6: np.float64(-0.020000000000000004)}
Motor Activation: -0.0738652205465518


This is with tanh but if we do matrix mult where for first layer like a nn we do from 1 to 2 but the next layer after we apply the nonlinear func we say 2 to 3 but here we do 1,2 to 3 instead.

In [120]:
# Here let's just say we have one hidden and call it a day
layer_one_nodes = [genome["nodes"][i] for i in range(0, 4)]
layer_three_nodes = [genome["nodes"][4]]
layer_two_nodes = [genome["nodes"][i] for i in range(5, 7)]

print("Layer 1 Nodes:", layer_one_nodes)
print("Layer 2 Nodes:", layer_two_nodes)
print("Layer 3 Nodes:", layer_three_nodes)

Layer 1 Nodes: [{'type': 'INPUT', 'act': 'identity'}, {'type': 'INPUT', 'act': 'identity'}, {'type': 'INPUT', 'act': 'identity'}, {'type': 'INPUT', 'act': 'identity'}]
Layer 2 Nodes: [{'type': 'HIDDEN', 'act': 'identity'}, {'type': 'HIDDEN', 'act': 'identity'}]
Layer 3 Nodes: [{'type': 'OUTPUT', 'act': 'tanh'}]


In [121]:
weights_one_two = np.zeros((len(layer_one_nodes), len(layer_two_nodes)))
# Doing +1 since I messed this up and just trying to test at this point since we need 7 indicies since if we every check for 4 but only have 6 indicies dont work
weights_two_three = np.zeros((len(layer_two_nodes) + weights_one_two.shape[0] + 1, len(layer_three_nodes)))

print(weights_one_two)
print(weights_two_three)

[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [122]:
for key in genome["connections"]:
    conn = genome["connections"][key]
    if not conn["enabled"]:
        continue

    in_node = conn["in"]
    out_node = conn["out"]

    try:
        # did -4 since for example it goes from 2 to 5
        # then for outputs of 5 and 6 we should have outputs there
        # just forcing it to work to test in real code it would be diff
        if out_node in [5, 6]:  # Hidden layer
            weights_one_two[in_node, out_node - 5] = conn["weight"]
    except:
        pass

    try:
        if out_node == 4:  # Output layer
            weights_two_three[in_node, out_node - 4] = conn["weight"]
    except:
        pass


print(weights_one_two)
print(weights_two_three)

[[ 0.   0. ]
 [-0.2 -0.2]
 [ 1.   0. ]
 [ 0.   0. ]]
[[ 0. ]
 [ 0. ]
 [ 0. ]
 [-0.5]
 [ 0. ]
 [ 0.4]
 [-0.2]]


In [123]:
middle_layer = inputs @ weights_one_two
# Identity so we gonna keep same
middle_inputs = np.concatenate((inputs, [0], middle_layer))
last_layer = middle_inputs @ weights_two_three
print(last_layer)

[-0.074]


This works and when I do multiple pendulums I can basically broadcast this so instead of 5 inputs I have 5 * total_pendulums which gives you info for all of them. So we can basically run all pendulums at once. Onlike for loops for each once.

And I'm not sure but if I did an n by n array I should be able to do recursive elements right?

### Neat part now

See what I did there

In [3]:
g = 9.81
l = 1.0
m = 1.0

init_x = 0.0
init_v = 0.0
init_theta = np.pi / 2
init_omega = 0.0

dt = 0.02
steps = 1000

n = 100  # Gonna be how many pendulums I'm going to spawn

#### First Initial Population

In [4]:
population = []
base_nodes = {
    0: {"type": "INPUT", "act": "identity"},  # x
    1: {"type": "INPUT", "act": "identity"},  # v
    2: {"type": "INPUT", "act": "identity"},  # θ
    3: {"type": "INPUT", "act": "identity"},  # ω
    4: {"type": "OUTPUT", "act": "tanh"},  # Motor Force
}
starting_connections = [
    {"id": 100, "in": 0, "out": 4},
    {"id": 101, "in": 1, "out": 4},
    {"id": 102, "in": 2, "out": 4},
    {"id": 103, "in": 3, "out": 4},
]

In [5]:
for _ in range(n):
    genome_nodes = {
        node_id: data.copy() for node_id, data in base_nodes.items()
    }  # To not pass by reference

    genome_connections = {}
    for conn in starting_connections:
        genome_connections[conn["id"]] = {
            "in": conn["in"],
            "out": conn["out"],
            "weight": np.random.uniform(-1.0, 1.0),  # Random start weight
            "enabled": True,
        }

    genome = {
        "nodes": genome_nodes,
        "connections": genome_connections,
        "fitness": 0.0,
        "species": None,
    }

    population.append(genome)

In [6]:
pprint.pprint(population[0])

{'connections': {100: {'enabled': True,
                       'in': 0,
                       'out': 4,
                       'weight': 0.4639039590987888},
                 101: {'enabled': True,
                       'in': 1,
                       'out': 4,
                       'weight': -0.035300944341165374},
                 102: {'enabled': True,
                       'in': 2,
                       'out': 4,
                       'weight': -0.19217263775194926},
                 103: {'enabled': True,
                       'in': 3,
                       'out': 4,
                       'weight': -0.7176480298380714}},
 'fitness': 0.0,
 'nodes': {0: {'act': 'identity', 'type': 'INPUT'},
           1: {'act': 'identity', 'type': 'INPUT'},
           2: {'act': 'identity', 'type': 'INPUT'},
           3: {'act': 'identity', 'type': 'INPUT'},
           4: {'act': 'tanh', 'type': 'OUTPUT'}},
 'species': None}


#### Evaluate

In [9]:
def compute_topo_order(genome):
    nodes = genome["nodes"]
    connections = genome["connections"]

    in_degree = {node_id: 0 for node_id in nodes}
    adj_list = {node_id: [] for node_id in nodes}
    for conn in connections.values():
        if conn["enabled"]:
            in_degree[conn["out"]] += 1
            adj_list[conn["in"]].append(conn["out"])

    queue = [node_id for node_id, degree in in_degree.items() if degree == 0]
    topo_order = []

    while queue:
        current = queue.pop(0)
        topo_order.append(current)

        for neighbor in adj_list[current]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    return topo_order

In [13]:
def solve_dag(genome, topo_order, values):
    node_values = {node_id: 0.0 for node_id in genome["nodes"]}

    for i, val in enumerate(values):
        node_values[i] = val

    for node_id in topo_order:
        node_data = genome["nodes"][node_id]

        if node_data["type"] != "INPUT":
            raw_val = node_values[node_id]
            if node_data["act"] == "tanh":
                node_values[node_id] = np.tanh(raw_val)
            elif node_data["act"] == "identity":
                node_values[node_id] = raw_val

        for conn in genome["connections"].values():
            if conn["enabled"] and conn["in"] == node_id:
                node_values[conn["out"]] += (
                    node_values[node_id] * conn["weight"]
                )

    return node_values[4] # 4 is the motor output

In [14]:
def run_simulation(genome, topo_order):
    x, v, theta, omega = init_x, init_v, init_theta, init_omega
    fitness = 0.0

    for _ in range(steps):
        motor_force = solve_dag(genome, topo_order, (x, v, theta, omega))

        alpha = (g * np.sin(theta) - motor_force * np.cos(theta)) / l
        omega += alpha * dt
        theta += omega * dt
        theta = ((theta + np.pi) % (2 * np.pi)) - np.pi
        v += motor_force * dt
        x += v * dt

        angle_reward = np.pi - abs(theta) # Max is 3.14 at top and you just lose points closer to bottom
        position_penalty = 0.1 * abs(x) # Lose points further from center
        fitness += angle_reward - position_penalty

    return fitness

In [15]:
for index, genome in enumerate(population):
    # A. Dynamically compute the topological order for this genome
    # (For Gen 0, this will just be inputs first, then output node 4)
    topo_order = compute_topo_order(genome)
    
    # B. Pass the genome and its order into your simulation
    # This runs the cart-pole physics and returns a float score
    fitness_score = run_simulation(genome, topo_order)
    
    # C. Store the score directly inside the genome dictionary
    genome["fitness"] = fitness_score
    
    # Print a status update every 25 genomes just to track progress
    if index % 25 == 0:
        print(f"  -> Evaluated {index}/{len(population)} genomes. Latest Fitness: {fitness_score:.2f}")

print("Evaluation complete! Every genome now has a fitness score.")

  -> Evaluated 0/100 genomes. Latest Fitness: -3489.78
  -> Evaluated 25/100 genomes. Latest Fitness: -1137.85
  -> Evaluated 50/100 genomes. Latest Fitness: 507.11
  -> Evaluated 75/100 genomes. Latest Fitness: 411.60
Evaluation complete! Every genome now has a fitness score.


#### Displaying

In [21]:
all_fitness = [genome["fitness"] for genome in population]
best_genome = np.argmax(all_fitness)
print(best_genome)
pprint.pprint(population[best_genome])

19
{'connections': {100: {'enabled': True,
                       'in': 0,
                       'out': 4,
                       'weight': -0.9608245343420736},
                 101: {'enabled': True,
                       'in': 1,
                       'out': 4,
                       'weight': 0.423171275601006},
                 102: {'enabled': True,
                       'in': 2,
                       'out': 4,
                       'weight': -0.598344388244155},
                 103: {'enabled': True,
                       'in': 3,
                       'out': 4,
                       'weight': 0.6824252431059146}},
 'fitness': np.float64(1626.8376256469248),
 'nodes': {0: {'act': 'identity', 'type': 'INPUT'},
           1: {'act': 'identity', 'type': 'INPUT'},
           2: {'act': 'identity', 'type': 'INPUT'},
           3: {'act': 'identity', 'type': 'INPUT'},
           4: {'act': 'tanh', 'type': 'OUTPUT'}},
 'species': None}


In [20]:
def plot_simulation_ghosting(
    history: list[dict[str, float]],
    pole_length: float = l,
    title: str = "Inverted Pendulum Ghosting",
):
    fig = go.Figure()

    fig.add_shape(
        type="line",
        x0=-5,
        y0=0,
        x1=5,
        y1=0,
        line=dict(color="Gray", width=1, dash="dash"),
    )

    for i, entry in enumerate(history):
        total_ghosts = len(history)

        cart_x = entry["x"]
        theta = entry["theta"]

        pole_x = cart_x + pole_length * np.sin(theta)
        pole_y = pole_length * np.cos(theta)

        opacity = max(0.1, i / total_ghosts)

        fig.add_trace(
            go.Scatter(
                x=[cart_x, pole_x],
                y=[0, pole_y],
                mode="lines+markers",
                line=dict(width=2, color="blue"),
                marker=dict(size=4, color="red"),
                opacity=opacity,
                showlegend=False,
                hoverinfo="text",
                text=f"Frame: {entry['frame']} | Theta: {np.degrees(theta):.1f}°",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[cart_x],
                y=[0],
                mode="markers",
                marker=dict(symbol="square", size=10, color="black"),
                opacity=opacity,
                showlegend=False,
            )
        )

    fig.update_layout(
        title=title,
        xaxis=dict(title="Cart Position (x)", range=[-2.5, 2.5]),
        yaxis=dict(
            title="Height (y)", range=[-1.5, 1.5], scaleanchor="x", scaleratio=1
        ),
        template="plotly_white",
        width=800,
        height=500,
    )

    return fig

In [26]:
history: list[dict[str, float]] = []

genome = population[best_genome]
topo_order = compute_topo_order(genome)
x, v, theta, omega = init_x, init_v, init_theta, init_omega

for _ in range(steps):
    node_values = {node_id: 0.0 for node_id in genome["nodes"]}

    for i, val in enumerate((x, v, theta, omega)):
        node_values[i] = val

    for node_id in topo_order:
        node_data = genome["nodes"][node_id]

        if node_data["type"] != "INPUT":
            raw_val = node_values[node_id]
            if node_data["act"] == "tanh":
                node_values[node_id] = np.tanh(raw_val)
            elif node_data["act"] == "identity":
                node_values[node_id] = raw_val

        for conn in genome["connections"].values():
            if conn["enabled"] and conn["in"] == node_id:
                node_values[conn["out"]] += node_values[node_id] * conn["weight"]

    motor_force = node_values[4]

    alpha = (g * np.sin(theta) - motor_force * np.cos(theta)) / l
    omega += alpha * dt
    theta += omega * dt
    theta = ((theta + np.pi) % (2 * np.pi)) - np.pi
    v += motor_force * dt
    x += v * dt

    history.append(
        {
            "x": x,
            "theta": theta,
            "node_values": node_values.copy(),  # Essential to .copy() so it doesn't overwrite
        }
    )

In [24]:
def animate_simulation(
    history: list[dict[str, float]],
    pole_length: float = 1.0,
    title: str = "Inverted Pendulum Animation",
):
    start = history[0]
    fig = go.Figure(
        data=[
            go.Scatter(
                x=[start["x"]],
                y=[0],
                mode="markers",
                marker=dict(symbol="square", size=20, color="black"),
                name="Cart",
            ),
            go.Scatter(
                x=[start["x"], start["x"] + pole_length * np.sin(start["theta"])],
                y=[0, pole_length * np.cos(start["theta"])],
                mode="lines+markers",
                line=dict(width=4, color="blue"),
                marker=dict(size=8, color="red"),
                name="Pole",
            ),
        ],
        layout=go.Layout(
            xaxis=dict(range=[-2.5, 2.5], autorange=False),
            yaxis=dict(
                range=[-1.5, 1.5], autorange=False, scaleanchor="x", scaleratio=1
            ),
            title=title,
            template="plotly_white",
            width=800,
            height=500,
            updatemenus=[
                dict(
                    type="buttons",
                    buttons=[
                        dict(
                            label="Play",
                            method="animate",
                            args=[
                                None,
                                {
                                    "frame": {"duration": 20, "redraw": False},
                                    "fromcurrent": True,
                                },
                            ],
                        ),
                        dict(
                            label="Pause",
                            method="animate",
                            args=[
                                [None],
                                {
                                    "frame": {"duration": 0, "redraw": False},
                                    "mode": "immediate",
                                    "fromcurrent": True,
                                },
                            ],
                        ),
                    ],
                )
            ],
        ),
        frames=[
            go.Frame(
                data=[
                    go.Scatter(x=[e["x"]], y=[0]),
                    go.Scatter(
                        x=[e["x"], e["x"] + pole_length * np.sin(e["theta"])],
                        y=[0, pole_length * np.cos(e["theta"])],
                    ),
                ],
                name=str(i),
            )
            for i, e in enumerate(history)
        ],
    )

    return fig

In [23]:
plot_simulation_ghosting(history[::5])

In [25]:
animate_simulation(history)

In [ ]:
# Lowkey can't lie most of this is ai gang
def animate_sim_with_brain(history: list[dict], genome: dict, pole_length: float = 1.0):
    node_positions = {}

    # Inputs go on the left column
    inputs = [nid for nid, ndata in genome["nodes"].items() if ndata["type"] == "INPUT"]
    for idx, nid in enumerate(sorted(inputs)):
        node_positions[nid] = (3.5, 0.6 - idx * 0.4)

    outputs = [
        nid for nid, ndata in genome["nodes"].items() if ndata["type"] == "OUTPUT"
    ]
    for idx, nid in enumerate(sorted(outputs)):
        node_positions[nid] = (5.5, 0.0)

    hiddens = [
        nid for nid, ndata in genome["nodes"].items() if ndata["type"] == "HIDDEN"
    ]
    for idx, nid in enumerate(sorted(hiddens)):
        node_positions[nid] = (4.5, 0.5 - idx * 0.5)

    fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5])
    start = history[0]

    # Trace 0: Cart
    fig.add_trace(
        go.Scatter(
            x=[start["x"]],
            y=[0],
            mode="markers",
            marker=dict(symbol="square", size=20, color="black"),
            name="Cart",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[start["x"], start["x"] + pole_length * np.sin(start["theta"])],
            y=[0, pole_length * np.cos(start["theta"])],
            mode="lines+markers",
            line=dict(width=4, color="blue"),
            marker=dict(size=8, color="red"),
            name="Pole",
        ),
        row=1,
        col=1,
    )

    conn_keys = list(genome["connections"].keys())
    for cid in conn_keys:
        conn = genome["connections"][cid]
        if conn["enabled"]:
            p1 = node_positions[conn["in"]]
            p2 = node_positions[conn["out"]]
            fig.add_trace(
                go.Scatter(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    mode="lines",
                    line=dict(width=2, color="gray"),
                    hoverinfo="none",
                    showlegend=False,
                ),
                row=1,
                col=2,
            )

    node_x = [pos[0] for pos in node_positions.values()]
    node_y = [pos[1] for pos in node_positions.values()]
    node_labels = [f"Node {nid}" for nid in node_positions.keys()]
    fig.add_trace(
        go.Scatter(
            x=node_x,
            y=node_y,
            mode="markers+text",
            text=node_labels,
            textposition="top center",
            marker=dict(size=14, color="darkcyan"),
            name="Nodes",
        ),
        row=1,
        col=2,
    )

    frames = []
    for frame_idx, entry in enumerate(history):
        frame_data = []

        frame_data.append(go.Scatter(x=[entry["x"]], y=[0]))
        frame_data.append(
            go.Scatter(
                x=[entry["x"], entry["x"] + pole_length * np.sin(entry["theta"])],
                y=[0, pole_length * np.cos(entry["theta"])],
            )
        )

        for cid in conn_keys:
            conn = genome["connections"][cid]
            if conn["enabled"]:
                source_node_val = entry["node_values"].get(conn["in"], 0.0)
                signal = source_node_val * conn["weight"]

                color_val = (
                    "green"
                    if signal > 0.1
                    else ("red" if signal < -0.1 else "lightgray")
                )
                width_val = min(
                    1 + abs(signal) * 3, 6
                ) 

                frame_data.append(
                    go.Scatter(line=dict(color=color_val, width=width_val))
                )

        frame_data.append(go.Scatter(x=node_x, y=node_y))

        frames.append(go.Frame(data=frame_data, name=str(frame_idx)))

    fig.frames = frames
    fig.update_layout(
        title="Cart-Pole Control & Neural Signal Telemetry",
        template="plotly_white",
        width=1000,
        height=500,
        xaxis=dict(range=[-2.5, 2.5], autorange=False),
        yaxis=dict(range=[-1.5, 1.5], autorange=False, scaleanchor="x", scaleratio=1),
        xaxis2=dict(
            range=[3.0, 6.0],
            autorange=False,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
        ),
        yaxis2=dict(
            range=[-1.5, 1.5],
            autorange=False,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
        ),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[
                            None,
                            {
                                "frame": {"duration": 20, "redraw": True},
                                "fromcurrent": True,
                            },
                        ],
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[
                            [None],
                            {
                                "frame": {"duration": 0, "redraw": False},
                                "mode": "immediate",
                                "fromcurrent": True,
                            },
                        ],
                    ),
                ],
            )
        ],
    )

    return fig

In [32]:
animate_sim_with_brain(history, genome)

#### Speciation

Basically a genome that tries new things will initally have random weights and do pretty bad. Speciation divides your population into distinct groups based on how topologically similar they are. Then we compete within the species. Then I think we mate.

In [ ]:
def get_compatibility_distance(genome1, genome2):
    # These weights make the differences between species
    c1 = 1.0  # Importance of disjoint/excess structural genes
    c3 = 0.4  # Importance of weight differences

    genes1 = set(genome1["connections"].keys()) # So all the connection numbers
    genes2 = set(genome2["connections"].keys())

    matching_genes = genes1.intersection(genes2) # Connection numbers in both like inital and output
    disjoint_excess = genes1.symmetric_difference(genes2) # Everything but union

    weight_diff = 0.0
    for key in matching_genes:
        weight_diff += abs(genome1["connections"][key]["weight"] - genome2["connections"][key]["weight"])
    avg_weight_diff = weight_diff / len(matching_genes) if matching_genes else 0.0

    # Normalize with size of larger gene
    N = max(len(genes1), len(genes2), 1)

    # NEAT Distance Formula
    distance = (c1 * len(disjoint_excess) / N) + (c3 * avg_weight_diff)
    return distance

In [40]:
def speciate(population, threshold=3.0):
    species_list = []

    for genome in population:
        assigned = False

        for spec in species_list:
            distance = get_compatibility_distance(genome, spec["representative"]) or 0

            if distance < threshold:
                spec["members"].append(genome)
                genome["species"] = id(spec)
                assigned = True
                break

        if not assigned:
            new_species = {
                "representative": genome,
                "members": [genome],
            }
            species_list.append(new_species)
            genome["species"] = id(new_species)

    species_list = [s for s in species_list if len(s["members"]) > 0]

    return species_list

In [42]:
species = speciate(population)
print(
    f"Speciation complete! Organized {len(population)} genomes into {len(species)} distinct species."
)
for idx, s in enumerate(species):
    print(f"  -> Species {idx}: Count = {len(s['members'])}")

Speciation complete! Organized 100 genomes into 1 distinct species.
  -> Species 0: Count = 100
